# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [1]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [2]:
y_value = df['loan_status'] # set the y
y_values = np.ravel(y_value) # change to an array (list)

x_values = df.drop('loan_status', axis=1) # drop the y from the dataframe

print(x_values.shape)
print(y_values.shape)

(50000, 15)
(50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [3]:
# split data into training and test
from sklearn.model_selection  import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(x_values, y_values, test_size = 0.2, random_state=4567, stratify=y_values)

# print the shapes to check everything is OK
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(40000, 15)
(10000, 15)
(40000,)
(10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [4]:
from sklearn.preprocessing import MinMaxScaler

# create a MinMaxScaler object
scaler = MinMaxScaler()

# fit the scaler on the TRAINING data only, then transform both sets
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=x_values.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=x_values.columns)
X_train_scaled.head()

,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio
0,0.134615,0.060150,0.131123,0.513944,0.176667,0.000253,0.061617,0.0,0.125,0.0,0.025126,0.498235,0.274436,0.028600,0.028614
1,0.038462,0.025063,0.213136,0.501992,0.046667,0.000630,0.047255,0.0,0.000,0.0,0.683417,0.791765,0.146617,0.523833,0.524096
2,0.134615,0.037594,0.102413,0.521912,0.063333,0.000313,0.044683,0.0,0.000,0.0,0.723618,0.470588,0.233083,0.927245,0.927711
3,0.211538,0.225564,0.101583,0.826693,0.013333,0.000277,0.060882,0.0,0.000,0.0,0.174874,0.551765,0.319549,0.226794,0.225904
4,0.115385,0.042607,0.023060,0.396414,0.170000,0.000097,0.061274,0.0,0.000,0.0,0.294472,0.550588,0.615288,0.728048,0.727410


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [5]:
from sklearn.linear_model import LogisticRegression as LogR
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn.ensemble import GradientBoostingClassifier as GBDT

LogR_algo = LogR()
LogR_model = LogR_algo.fit(X_train_scaled, Y_train) # scaled predictors

RF_algo = RF()
RF_model = RF_algo.fit(X_train, Y_train) # unscaled predictors

GBDT_algo = GBDT()
GBDT_model = GBDT_algo.fit(X_train, Y_train) # unscaled predictors

models = [LogR_model, RF_model, GBDT_model]
names = ['Logistic Regression', 'Random Forest', 'GBDT']

## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

train_sets = [X_train_scaled, X_train, X_train]
test_sets = [X_test_scaled, X_test, X_test]

for i in range(3):
  print(f"Model: {names[i]}")

  for label, X, Y in [('Train', train_sets[i], Y_train), ('Test', test_sets[i], Y_test)]:
    predict = models[i].predict(X)

    # Calculate accuracy, precision, recall, and F1-score
    precision, recall, f1_score, _ = precision_recall_fscore_support(Y, predict, average='macro')
    print(f"{label} Accuracy: {round(accuracy_score(Y, predict),3)}")
    print(f"{label} Macro Precision: {round(precision,3)}")
    print(f"{label} Macro Recall: {round(recall,3)}")
    print(f"{label} Macro F1-score: {round(f1_score,3)}")
  print("\n")

Model: Logistic Regression
Train Accuracy: 0.838
Train Macro Precision: 0.837
Train Macro Recall: 0.835
Train Macro F1-score: 0.836
Test Accuracy: 0.841
Test Macro Precision: 0.841
Test Macro Recall: 0.838
Test Macro F1-score: 0.839


Model: Random Forest
Train Accuracy: 1.0
Train Macro Precision: 1.0
Train Macro Recall: 1.0
Train Macro F1-score: 1.0
Test Accuracy: 0.885
Test Macro Precision: 0.885
Test Macro Recall: 0.884
Test Macro F1-score: 0.884


Model: GBDT
Train Accuracy: 0.891
Train Macro Precision: 0.892
Train Macro Recall: 0.889
Train Macro F1-score: 0.89
Test Accuracy: 0.89
Test Macro Precision: 0.89
Test Macro Recall: 0.887
Test Macro F1-score: 0.889




## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?